# Lab: End-to-End Generalized Graph RAG
In this lab, we build a fully dynamic Graph RAG pipeline. It reads a PDF, extracts entities and relationships dynamically, builds an in-memory knowledge graph, and traverses the graph to answer user questions with objective explainability.

### Step 1: Install Dependencies

In [ ]:
!pip install networkx requests PyPDF2 matplotlib


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Import Libraries

In [ ]:
import os
import json
import requests
import PyPDF2
import networkx as nx
import matplotlib.pyplot as plt

### Step 3: Setup API Keys & LLM Configuration

In [ ]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key: ").strip()
else:
    print("Success: API Key loaded!")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
TEXT_MODEL = "openai/gpt-oss-20b:free"

### Step 4: Download Target Document

In [51]:
PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"

os.makedirs("data", exist_ok=True)
PDF_PATH = os.path.join("data", "document.pdf")

try:
    print("Downloading document...")
    response = requests.get(PDF_URL)
    response.raise_for_status()
    
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)
        
    print(f"Success! PDF stored at: {PDF_PATH}")
except Exception as e:
    print(f"Error downloading PDF: {e}")

Success! PDF stored at: data\document.pdf


### Step 5: Extract Text via PyPDF2

In [52]:
sample_text = ""

try:
    with open(PDF_PATH, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        page = reader.pages[0]
        sample_text = page.extract_text()
    
    # We take the first 1500 characters to keep our LLM API payload fast
    sample_text = sample_text[:1500].strip()
    
    print(f"Success! Extracted {len(sample_text)} characters.")
except Exception as e:
    print(f"Error reading PDF file: {e}")

Success! Extracted 1500 characters.


### Step 6: Domain-Agnostic Entity Extraction

In [53]:
def extract_graph_elements(text):
    """Uses LLM to convert raw text into structured Nodes and Edges."""
    
    prompt = f"""
    You are an expert knowledge graph builder. Read the text below and extract all key entities and their relationships.

    Text:
    {text}

    CRITICAL INSTRUCTIONS:
    Output ONLY a valid JSON list of objects with keys "source", "relation", and "target".
    Example format:
    [
      {{"source": "Entity_A", "relation": "RELATES_TO", "target": "Entity_B"}},
      {{"source": "Entity_C", "relation": "CAUSES", "target": "Entity_D"}}
    ]
    Do not add any Markdown code blocks, explanations, or introductory text. Return ONLY pure JSON.
    """
    
    payload = {
        "model": TEXT_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    
    try:
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        resp.raise_for_status()
        raw_json = resp.json()["choices"][0]["message"]["content"].strip()
        
        # Clean up unexpected markdown
        if raw_json.startswith("```"):
            raw_json = raw_json.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            
        return json.loads(raw_json)
    except Exception as e:
        print(f"Extraction Error: {e}")
        return []

extracted_relationships = extract_graph_elements(sample_text)
print(f"Extracted {len(extracted_relationships)} relationships:\n")
print(json.dumps(extracted_relationships, indent=2))

Extracted 31 relationships:

[
  {
    "source": "Google",
    "relation": "GRANTS_PERMISSION_TO",
    "target": "tables and figures"
  },
  {
    "source": "Google",
    "relation": "HAS_SUBUNIT",
    "target": "Google Brain"
  },
  {
    "source": "Google",
    "relation": "HAS_SUBUNIT",
    "target": "Google Research"
  },
  {
    "source": "Google Brain",
    "relation": "AFFILIATED_WITH",
    "target": "Google"
  },
  {
    "source": "Google Research",
    "relation": "AFFILIATED_WITH",
    "target": "Google"
  },
  {
    "source": "Aidan N. Gomez",
    "relation": "AFFILIATED_WITH",
    "target": "University of Toronto"
  },
  {
    "source": "Ashish Vaswani",
    "relation": "AFFILIATED_WITH",
    "target": "Google Brain"
  },
  {
    "source": "Noam Shazeer",
    "relation": "AFFILIATED_WITH",
    "target": "Google Brain"
  },
  {
    "source": "Niki Parmar",
    "relation": "AFFILIATED_WITH",
    "target": "Google Research"
  },
  {
    "source": "Jakob Uszkoreit",
    "relati

### Step 7: Build the Knowledge Graph

In [54]:
G = nx.DiGraph()

for rel in extracted_relationships:
    src = rel["source"].strip()
    tgt = rel["target"].strip()
    relation_type = rel["relation"].strip()
    
    G.add_edge(src, tgt, relation=relation_type)

print(f"Graph constructed successfully!")
print(f"Total Nodes: {G.number_of_nodes()}")
print(f"Total Edges: {G.number_of_edges()}")

Graph constructed successfully!
Total Nodes: 28
Total Edges: 30


### Step 8: Dynamic Traversal & Subgraph Retrieval

In [55]:
def find_node_in_question(question, graph):
    """Automatically finds which graph node the user is asking about."""
    for node in graph.nodes():
        if node.lower() in question.lower():
            return node
    return None

In [ ]:
def traverse_subgraph(graph, start_entity, radius=2):
    """Finds all relationships connected to an entity within N hops."""
    matching_nodes = [node for node in graph.nodes if start_entity.lower() in node.lower()]
    if not matching_nodes:
        return []

    target_node = matching_nodes[0]
    # BFS out to `radius` hops to pull in a local neighborhood of the entity
    sub_nodes = nx.single_source_shortest_path_length(graph, target_node, cutoff=radius).keys()
    subgraph = graph.subgraph(sub_nodes)

    facts = []
    for u, v, data in subgraph.edges(data=True):
        fact = f"{u} --[{data.get('relation', 'CONNECTED_TO')}]--> {v}"
        facts.append(fact)

    return facts

### Step 9: The Generalized RAG Engine

In [ ]:
def execute_graph_rag(question):
    """A fully generalized Graph RAG pipeline."""

    # Step 1: Identify which known graph entity the question refers to
    target_entity = find_node_in_question(question, G)

    if not target_entity:
        return f"Could not find any known concepts in your question. Known concepts: {list(G.nodes)[:5]}..."

    print(f"[System Log] Auto-detected focus concept: '{target_entity}'")

    # Step 2: Retrieve connected facts (subgraph) around the detected entity
    retrieved_facts = traverse_subgraph(G, target_entity, radius=2)
    if not retrieved_facts:
        return f"Found the concept '{target_entity}', but no relationships are connected to it."

    facts_block = "\n".join([f"- {f}" for f in retrieved_facts])

    # Step 3: Build a grounded prompt so the LLM answers only from retrieved graph facts
    prompt = f"""
    You are an expert AI research assistant using a Knowledge Graph.
    Answer the question using ONLY the connected relationship paths provided below.

    Graph Relationships:
    {facts_block}

    Question: {question}

    CRITICAL INSTRUCTIONS:
    Output your response in EXACTLY two sections as shown below.

    --- FINAL ANSWER ---
    [Provide a direct, simple, 1-sentence answer.]

    --- AI TRACING & EXPLAINABILITY ---
    [Explain step-by-step how the answer was derived from the graph. Use an objective, third-person perspective. Do NOT use first-person pronouns like "I" or "my".]
    """

    payload = {
        "model": TEXT_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}

    # Step 4: Call the LLM and return its answer
    try:
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"Error executing Graph RAG: {e}"

### Step 10: Test the Pipeline

In [58]:
query_1 = "What mechanism does the Transformer architecture rely on?"
print(execute_graph_rag(query_1))

[System Log] Auto-detected focus concept: 'Transformer'


--- FINAL ANSWER ---
The Transformer architecture relies on attention mechanisms.

--- AI TRACING & EXPLAINABILITY ---
The graph contains a direct relationship stating that the Transformer node has a [USES] edge pointing to the attention mechanisms node. This indicates that the Transformer architecture depends on attention mechanisms. No other mechanism is listed as a primary dependency in the provided relationships. Therefore, the answer is that the Transformer relies on attention mechanisms.
